# Final Individual Project: Data Check & Cleaning

**Dataset:** `YRBS_2007.csv` | **Target Audience:** US High School Students  
**Responsible Person:** 李宥宣 (`113370237`)  
---

### Objective
The primary goal of this notebook is to perform **Data Cleaning** and **Variable Recoding** using the selected variables for the Final Individual Project (ANOVA & Multiple Regression). We focus on:
1. **Loading** the raw dataset without modification.
2. **Extracting** the target variables (`WereThreatenedOrInjuredWithAWeaponOnSchoolProperty`, `BMIPCT`, and `AerobicExercise`).
3. **Handling** missing or invalid values.
4. **Recoding** the independent variable according to the required 3-group rules.
5. **Exporting** the processed master data for reproducibility.

### Research Questions & Variable Definitions

Following the Final Individual Project guidelines, we established our variables for ANOVA and Regression analyses:

**One-Way ANOVA & Multiple Linear Regression Analysis**
* **Research Question 1 (ANOVA):** Is there a significant difference in BMI percentiles among students based on the frequency they were threatened or injured with a weapon on school property?
* **Research Question 2 (Regression):** After controlling for aerobic exercise habits, does the frequency of weapon threats still significantly predict BMI percentiles?
* **Independent Variable (Group):** `WereThreatenedOrInjuredWithAWeaponOnSchoolProperty` (Weapon threat frequency)
* **Dependent Variable (Continuous):** `BMIPCT` (BMI Percentile)
* **Control Variable (Continuous/Ordinal):** `AerobicExercise` (Frequency/Days of aerobic exercise)
* **Recoding Rule for Independent Variable (`Threat_Group`):**
  * **Group 1 ('0 times'):** code 1 (0 times) 
  * **Group 2 ('1 time'):** code 2 (1 time)
  * **Group 3 ('2+ times'):** codes 3 through 8 (2 or more times)

### Data Processing Pipeline

To ensure **reproducibility**, this cleaning script follows these steps:
1. **Load Data:** Read `../data/raw/YRBS_2007.csv`.
2. **Subset:** Extract only `WereThreatenedOrInjuredWithAWeaponOnSchoolProperty`, `BMIPCT`, and `AerobicExercise`.
3. **Handle Missing Data:** Count and drop rows with `NaN` in any of the selected columns.
4. **Recoding:** Convert the threat variable into a 3-category variable (`Threat_Group`) strictly following the recoding rules.
5. **Data Validation:** Check the summary statistics and sample sizes of the three groups.
6. **Export:** Save the final clean dataset to `../data/processed/cleaned_yrbs_master.csv` for the inference phase.

In [23]:
# Import pandas for data manipulation and analysis
import pandas as pd
# Import numpy for numerical operations, specifically handling NaN values
import numpy as np
# Import os module to interact with the operating system
import os

# ==========================================================
# 1. Environment Setup and Loading
# ==========================================================
# Define the relative file paths
raw_data_path = '../data/raw/YRBS_2007.csv'
processed_data_path = '../data/processed/cleaned_yrbs_master.csv'

# Create the 'processed' directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Load the raw CSV file into a pandas DataFrame
df_raw = pd.read_csv(raw_data_path)
initial_count = len(df_raw)
print(f"✅ Raw data loaded successfully. Initial sample size: {initial_count}")

# ==========================================================
# 2. Variable Selection 
# ==========================================================
# Define the variables
iv_var = 'WereThreatenedOrInjuredWithAWeaponOnSchoolProperty'
dv_var = 'BMIPCT'
control_var = 'AerobicExercise'

# Create a new DataFrame containing only the required columns
df_selected = df_raw[[iv_var, dv_var, control_var]].copy()

# ==========================================================
# 3. Handling Missing Values (NaN)
# ==========================================================
print("\n[Step 1] Missing Value Statistics:")
print(df_selected.isnull().sum().to_string())

# Drop rows that contain missing values in any of the columns
df_cleaned = df_selected.dropna().copy()
after_dropna_count = len(df_cleaned)
print(f"\n[Step 2] Sample size after dropping missing values: {after_dropna_count}")

# ==========================================================
# 4. Variable Recoding
# ==========================================================
# Define a custom function to recode the survey responses into 3 categories
def recode_threats(code):
    if code == 1:
        return '0 times'
    elif code == 2:
        return '1 time'
    elif 3 <= code <= 8:
        return '2+ times'
    else:
        return np.nan # Invalid codes converted to NaN

# Apply the recoding function
new_col_name = 'Threat_Group'
df_cleaned[new_col_name] = df_cleaned[iv_var].apply(recode_threats)

# Drop any rows that became NaN during the recoding process
df_final = df_cleaned.dropna().copy()
final_count = len(df_final)

print(f"\n[Step 3] Recoding Verification ({new_col_name}):")
print(df_final[new_col_name].value_counts(dropna=False).to_string())

# ==========================================================
# 5. Data Validation & Summary Report
# ==========================================================
print("\n" + "="*50)
print(f"🏆 Final Data Cleaning Summary:")
print(f"   - Initial raw samples: {initial_count}")
print(f"   - Final usable samples: {final_count}")
print(f"   - Total attrition rate: {((initial_count - final_count) / initial_count * 100):.2f}%")
print("-" * 50)
print(f"   📊 Group Breakdown (Ready for ANOVA & Regression):")
group_counts = df_final[new_col_name].value_counts()
print(f"      * Group '0 times':  {group_counts.get('0 times', 0)}")
print(f"      * Group '1 time':   {group_counts.get('1 time', 0)}")
print(f"      * Group '2+ times': {group_counts.get('2+ times', 0)}")
print("="*50)

# Export the completely clean DataFrame to a CSV file
df_final.to_csv(processed_data_path, index=False)
print(f"\n💾 Processed master dataset saved to: {processed_data_path}")

✅ Raw data loaded successfully. Initial sample size: 14041

[Step 1] Missing Value Statistics:
WereThreatenedOrInjuredWithAWeaponOnSchoolProperty     147
BMIPCT                                                 979
AerobicExercise                                       1530

[Step 2] Sample size after dropping missing values: 11541

[Step 3] Recoding Verification (Threat_Group):
Threat_Group
0 times     10648
2+ times      496
1 time        397

🏆 Final Data Cleaning Summary:
   - Initial raw samples: 14041
   - Final usable samples: 11541
   - Total attrition rate: 17.80%
--------------------------------------------------
   📊 Group Breakdown (Ready for ANOVA & Regression):
      * Group '0 times':  10648
      * Group '1 time':   397
      * Group '2+ times': 496

💾 Processed master dataset saved to: ../data/processed/cleaned_yrbs_master.csv


In [25]:
# Define the comprehensive content for the reference document
reference_content = """# Final Individual Project: Variable Notes & Recoding Rules

**Dataset:** `YRBS_2007.csv`
**Research Question:** Weapon Threat Frequency and BMI Percentile (ANOVA & Regression)

## 1. Independent Variable Definition & Recoding
* **Variable Name:** `WereThreatenedOrInjuredWithAWeaponOnSchoolProperty`
* **Definition:** Used to divide the sample into three independent groups based on the frequency of being threatened with a weapon on school property.
* **Recoding Rules (`Threat_Group`):**
  * Original Code `1` (0 times) -> **Recoded as `0 times`**
  * Original Code `2` (1 time) -> **Recoded as `1 time`**
  * Original Codes `3` through `8` (2 or more times) -> **Recoded as `2+ times`**
  * *Note: Missing values (NaN) or any other invalid codes were dropped.*

## 2. Dependent Variable Definition
* **Variable Name:** `BMIPCT`
* **Definition:** Continuous response variable representing the student's BMI percentile.
* *Note: Missing values (NaN) were dropped.*

## 3. Control Variable Definition (For Multiple Regression)
* **Variable Name:** `AerobicExercise`
* **Definition:** Represents the frequency/days of aerobic exercise, used as a control variable to test the robustness of the weapon threat effect.
* *Note: Missing values (NaN) were dropped.*
"""

# Define the target directory and file path for the reference note
import os
reference_dir = '../references'
reference_file_path = os.path.join(reference_dir, 'variable_notes.md')

# Create the 'references' directory if it does not already exist
os.makedirs(reference_dir, exist_ok=True)

# Write the defined content into the markdown file
with open(reference_file_path, 'w', encoding='utf-8') as f:
    f.write(reference_content)

# Print a confirmation message to verify the file was saved successfully
print(f"✅ Reference documentation successfully created at: {reference_file_path}")

✅ Reference documentation successfully created at: ../references\variable_notes.md
